# FlashInspector AI – Test in Google Colab

**[→ Open in Colab](https://colab.research.google.com/github/patrisiyarum/fire/blob/main/flashinspector-ai/train_on_colab.ipynb)**

Run inference and view results. **Training is done in Kaggle** — use this notebook to test your trained `best.pt` model.

**Flow:** Get code → Install deps → Load model (Drive or upload) → Upload image/video → Run inference.

## 1. Check GPU

In [ ]:
!nvidia-smi 2>/dev/null || echo "No GPU (nvidia-smi not found). In Colab: Runtime → Change runtime type → T4 GPU."

## 2. Get the project code

**Option A – Clone from GitHub:** Set `REPO_URL` to your repo in the next cell and run it (leave `USE_UPLOADED = False`).

**Option B – Folder already in Colab:** If `flashinspector-ai` is already at a path (e.g. you unzipped it), set `USE_UPLOADED = True` and `UPLOADED_PATH` in the cell below and run.

**Option C – Upload zip (no GitHub):** Run the **"Optional - Option C"** cell below first: upload a zip of your `flashinspector-ai` folder. Then in the next cell set `USE_UPLOADED = True`, `UPLOADED_PATH = "/content/flashinspector-ai"` (or the path that appeared after unzip) and run.

In [ ]:
# OPTIONAL - Option C: Upload a zip of flashinspector-ai (no GitHub needed)
# 1. On your computer: zip the flashinspector-ai folder
# 2. Run this cell, click "Choose Files", select the zip
# 3. In the cell below, set USE_UPLOADED = True and run it

from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # opens file picker
for name in uploaded:
    if name.endswith(".zip"):
        Path("/content").mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(name, "r") as z:
            z.extractall("/content")
        print(f"Unzipped to /content (look for flashinspector-ai folder)")
        break

In [ ]:
# Choose ONE option:
# A) Clone from GitHub (set REPO_URL below)
# B) You already have the folder at UPLOADED_PATH (e.g. after unzipping)
# C) Upload a zip: run the next cell first to upload flashinspector-ai.zip, then set USE_UPLOADED = True

REPO_URL = "https://github.com/patrisiyarum/fire.git"  # for Option A: your repo URL
USE_UPLOADED = False  # True = use folder at UPLOADED_PATH (Options B or C)
UPLOADED_PATH = "/content/flashinspector-ai"

if USE_UPLOADED:
    import os
    PROJECT_DIR = UPLOADED_PATH
    assert os.path.isdir(PROJECT_DIR), f"Folder not found: {UPLOADED_PATH}. For Option C, run the upload cell below first."
    %cd $PROJECT_DIR
    print(f"Using project at {PROJECT_DIR}")
else:
    !git clone --depth 1 $REPO_URL /content/fire
    %cd /content/fire/flashinspector-ai
    PROJECT_DIR = "/content/fire/flashinspector-ai"
    print(f"Cloned and using {PROJECT_DIR}")

## 3. Install dependencies

You may see dependency conflict warnings; **ignore them** — inference will work.

In [ ]:
!pip install -q --upgrade-strategy only-if-needed ultralytics roboflow opencv-python python-dotenv pyyaml tqdm

## 4. Load model and run inference

**Model:** Copy `best.pt` from Kaggle Output to Google Drive at `My Drive/flashinspector_models/best.pt` (or upload in the quick path below).

**Input:** Upload an image/video (next cell), or use a YouTube URL.

In [ ]:
# Model path: Drive (My Drive/flashinspector_models/best.pt) or upload in quick path below
from pathlib import Path
import os

# Ensure we're in the project dir (needed if you run section 8 without running section 2 first)
for d in [".", "/content/fire/flashinspector-ai", "/content/flashinspector-ai"]:
    if (Path(d) / "fire_safety_datasets").exists():
        os.chdir(Path(d).resolve())
        print(f"Working dir: {os.getcwd()}")
        break

# Prefer merged model (runs/), then single-dataset (fire_safety_models/)
MODEL_PATH = Path("runs/fire_safety/weights/best.pt")
if not MODEL_PATH.exists():
    MODEL_PATH = Path("fire_safety_models/firenet_nano/weights/best.pt")
if not MODEL_PATH.exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_dir = Path("/content/drive/MyDrive/flashinspector_models")
    MODEL_PATH = drive_dir / "best.pt"
    if not MODEL_PATH.exists():
        # Search for best.pt in subfolders (e.g. firenet_nano/weights/)
        found = list(drive_dir.rglob("best.pt"))
        MODEL_PATH = found[0] if found else None
    if MODEL_PATH is None or not Path(MODEL_PATH).exists():
        raise FileNotFoundError("No best.pt found. Copy from Kaggle Output to Drive (My Drive/flashinspector_models/), or use the quick path below to upload.")
print(f"Using model: {MODEL_PATH}")

In [ ]:
# Upload an image or video to test (or skip and use a dataset image in the next cell)
from google.colab import files
uploaded = files.upload()  # click "Choose Files" and pick an image or video
TEST_IMAGE = list(uploaded.keys())[0] if uploaded else None
if TEST_IMAGE:
    print(f"Uploaded: {TEST_IMAGE}")

In [ ]:
# OR: Use a YouTube URL (set YOUTUBE_URL and run; skip if you already uploaded above)
YOUTUBE_URL = ""  # e.g. "https://www.youtube.com/watch?v=58naKHqpCWo"

if YOUTUBE_URL:
    import re
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"], check=True)
    import yt_dlp
    video_id = re.search(r"(?:v=|/)([a-zA-Z0-9_-]{11})(?:\?|&|$)", YOUTUBE_URL)
    video_id = video_id.group(1) if video_id else "video"
    youtube_out = Path.cwd() / f"youtube_video_{video_id}.mp4"
    ydl_opts = {"format": "best[ext=mp4]/best", "outtmpl": str(youtube_out), "quiet": False}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([YOUTUBE_URL])
        TEST_IMAGE = str(youtube_out)
        print(f"Downloaded to: {TEST_IMAGE}")
    except Exception as e:
        if "bot" in str(e).lower() or "Sign in" in str(e) or "cookies" in str(e).lower():
            print("YouTube blocked the download (bot check). Workarounds:")
            print("  1. Try a different video (some are not blocked).")
            print("  2. Download the video on your computer, then use the 'Upload' cell above and run inference on the uploaded file.")
            TEST_IMAGE = None  # so inference cell will ask for upload/dataset
        else:
            raise

In [ ]:
# Use uploaded image/video, or a dataset sample if you have one
from pathlib import Path

try:
    _ = TEST_IMAGE
except NameError:
    TEST_IMAGE = None
if not TEST_IMAGE:
    sample_dir = Path("fire_safety_datasets/firenet/valid/images")
    if sample_dir.exists():
        samples = list(sample_dir.glob("*.*"))[:1]
        TEST_IMAGE = str(samples[0]) if samples else None
    if not TEST_IMAGE:
        raise SystemExit("Upload an image in the cell above, or run section 5 (download dataset) first.")

# Find project dir (where test_model.py lives) - cwd can be /content in some sessions
PROJECT_DIR = None
for d in [Path.cwd(), Path("/content/fire/flashinspector-ai"), Path("/content/flashinspector-ai")]:
    script = d / "fire_safety_datasets" / "test_model.py"
    if script.exists():
        PROJECT_DIR = d
        break
if PROJECT_DIR is None:
    raise FileNotFoundError("Run section 2 (get project code) first. test_model.py not found.")

# Resolve path - uploaded/YouTube files can be in cwd or /content/
test_path = Path(TEST_IMAGE)
if not test_path.is_absolute():
    for base in [Path.cwd(), Path("/content"), PROJECT_DIR]:
        p = (base / test_path).resolve()
        if p.exists():
            test_path = p
            break
if not test_path.exists():
    raise FileNotFoundError(f"Image/video not found: {test_path}. Re-run the upload/YouTube cell above.")
TEST_IMAGE = str(test_path)
print(f"Project: {PROJECT_DIR}")
print(f"Model: {MODEL_PATH} (exists: {Path(MODEL_PATH).exists()})")
print(f"Input: {TEST_IMAGE} (exists: {Path(TEST_IMAGE).exists()})")

# Run inference (direct for images; script for video)
from ultralytics import YOLO
import cv2
import subprocess

out_dir = PROJECT_DIR / "inference_results"
out_dir.mkdir(exist_ok=True)
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

if Path(test_path).suffix.lower() in VIDEO_EXTS:
    # Video (including YouTube downloads): use script
    script_path = PROJECT_DIR / "fire_safety_datasets" / "test_model.py"
    r = subprocess.run(
        ["python", str(script_path), TEST_IMAGE, "--model", str(MODEL_PATH), "--conf", "0.25", "--frame-skip", "5"],
        cwd=str(PROJECT_DIR), capture_output=True, text=True
    )
    print(r.stdout)
    if r.stderr:
        print("STDERR:", r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"test_model.py failed (code {r.returncode}). Check output above.")
else:
    # Image: run directly (avoids shell path issues)
    model = YOLO(str(MODEL_PATH))
    results = model(test_path, conf=0.25, verbose=True)[0]
    out_path = out_dir / f"result_{Path(test_path).name}"
    cv2.imwrite(str(out_path), annotated := results.plot())
    print(f"Saved to: {out_path}")

### Transcribe video (optional)

When the input is a video, run this cell to extract and transcribe speech. Uses Whisper (free, runs in Colab).

In [ ]:
# Transcribe video (run after inference on a video)
from pathlib import Path
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
try:
    inp = Path(test_path) if "test_path" in dir() else Path(TEST_IMAGE)
except NameError:
    inp = None
if inp and inp.suffix.lower() in VIDEO_EXTS and inp.exists():
    !pip install -q openai-whisper 2>/dev/null
    !apt-get install -y ffmpeg 2>/dev/null || true
    import subprocess
    import whisper
    audio_path = Path("/tmp/audio_for_whisper.wav")
    subprocess.run(["ffmpeg", "-y", "-i", str(inp), "-vn", "-acodec", "pcm_s16le", "-ar", "16000", str(audio_path)], capture_output=True)
    if audio_path.exists():
        model = whisper.load_model("base")
        result = model.transcribe(str(audio_path))
        print("--- Transcript ---\n")
        print(result["text"])
    else:
        print("Could not extract audio. Install ffmpeg: !apt install -y ffmpeg")
else:
    print("Input is not a video, or TEST_IMAGE not set. Run the inference cells above first with a video.")

In [ ]:
# Show the result in the notebook (image = display; video = download link)
from IPython.display import Image, display
from pathlib import Path
from google.colab import files as colab_files

# Results are in project dir (PROJECT_DIR set by inference cell)
result_dir = PROJECT_DIR / "inference_results" if "PROJECT_DIR" in dir() else Path.cwd() / "inference_results"
name = Path(TEST_IMAGE).name
stem = Path(TEST_IMAGE).stem
result_path = result_dir / f"result_{name}"
if not result_path.exists():
    result_path = result_dir / f"result_{stem}.mp4"
if result_path.exists():
    if result_path.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"):
        display(Image(str(result_path), width=600))
    else:
        print(f"Annotated video saved: {result_path}")
        colab_files.download(str(result_path))
    print(f"Output: {result_path}")
else:
    print("Result not found. If the cell above showed an error, fix it and re-run.")
    print(f"Looked in: {result_dir}")
    if result_dir.exists():
        !ls -la "{result_dir}"
    else:
        print("inference_results/ was not created – test_model.py likely failed. Run section 2 (get code) first.")

## 5. Quick path: upload model + image

Minimal flow — clone, install, upload `best.pt` and an image. Skip sections 1–4 if you use this.

In [ ]:
# Setup: install deps, clone repo, get model
!pip install -q ultralytics opencv-python
!git clone --depth 1 https://github.com/patrisiyarum/fire.git /content/fire 2>/dev/null || true
%cd /content/fire/flashinspector-ai
PROJECT_DIR = "/content/fire/flashinspector-ai"

USE_DRIVE = False  # True = load from Drive (My Drive/flashinspector_models/best.pt)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    MODEL_PATH = "/content/drive/MyDrive/flashinspector_models/best.pt"
else:
    from google.colab import files
    print("Upload best.pt:")
    up = files.upload()
    MODEL_PATH = list(up.keys())[0]
print("Model:", MODEL_PATH)

In [ ]:
# Upload test image and run inference
from google.colab import files
from pathlib import Path
from ultralytics import YOLO
import cv2

print("Upload an image to test:")
img_up = files.upload()
img_path = list(img_up.keys())[0]

model = YOLO(MODEL_PATH)
results = model(img_path, conf=0.25)[0]
out_dir = Path(PROJECT_DIR) / "inference_results"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{Path(img_path).name}"
cv2.imwrite(str(out_path), results.plot())
print("Saved to:", out_path)

In [ ]:
# Show result
from IPython.display import Image, display
display(Image(str(out_path), width=600))